# physics-informed mri reconstruction colab

undersampled k-space -> high-fidelity image via an unrolled net with a data-consistency layer runs the whole pipeline on a colab accelerator smoke test -> train on synthetic -> fastmri -> evaluate + figures

> colab storage is ephemeral mount google drive and save the data subset checkpoints and figures there or they vanish when the runtime disconnects

set runtime to gpu via runtime -> change runtime type (t4/l4/a100/h100) see the table for which accelerator to pick

| accelerator | good for | notes |
|---|---|---|
| a100 / h100 | both projects fast | ampere/hopper tf32 + bf16 ideal for the unrolled net |
| l4 | both projects | ada 24 gb bf16/tf32 great value |
| t4 (free) | synthetic + small fastmri | 16 gb use amp_dtype fp16 (no bf16) |
| g4 / older gpu | synthetic + small runs | auto-detected via nvidia-smi |
| cpu | smoke test + synthetic | validates the pipeline too slow for real training |
| v5e-1 / v6e-1 tpu | project 2 only (maybe) | see the tpu note below |

> tpu note project 1 reconstruction is built on complex-valued ffts not reliably
> supported on tpu/xla use a gpu runtime for project 1 project 2 (real-valued
> segmentation) can in principle use a tpu but monai + torch_xla is finicky a gpu
> is still the smoother path

## 1. check the runtime / accelerator

In [ ]:
import os, subprocess, sys
print("Python:", sys.version.split()[0])
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
    ).decode().strip()
    print("GPU:", out)
except Exception:
    print("No NVIDIA GPU detected (CPU or TPU runtime).")
print("TPU env:", os.environ.get("COLAB_TPU_ADDR", "(none)"))

## 2. get the project code

pick one method below by setting SETUP
- github easiest once the repo is pushed (set REPO_URL)
- drive put the 01-mri-reconstruction folder in google drive (set PROJECT_DIR)
- upload a zip of the project

In [ ]:
SETUP = "github"   # "github" | "drive" | "upload"
REPO_URL = "https://github.com/cosmicquilt/mri-reconstruction.git"
PROJECT_DIR = "/content/mri-reconstruction"

import os
if SETUP == "github":
    if os.path.exists(PROJECT_DIR):
        get_ipython().system(f'git -C {PROJECT_DIR} pull')   # already cloned, pull latest
    else:
        get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
elif SETUP == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/job/projects/01-mri-reconstruction"  # set me
elif SETUP == "upload":
    from google.colab import files
    import zipfile, io
    up = files.upload()  # choose a project .zip
    name = next(iter(up))
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall("/content")
    PROJECT_DIR = "/content/" + name.replace(".zip", "")

os.chdir(PROJECT_DIR)
import sys
SRC = os.path.join(PROJECT_DIR, "src")
sys.path.insert(0, SRC)                          # in-kernel imports (import mri_recon ...)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")  # for !python cells
print("project dir:", os.getcwd())

## 3. install + smoke test

colab already ships a cuda build of pytorch so install only the extra deps (torch left untouched) the code is already importable via the path set in the previous cell no editable install needed (that needs a kernel restart on colab)

In [ ]:
!pip -q install pyyaml h5py scikit-image matplotlib
!python scripts/smoke_test.py

## 4. device + gpu speedups

In [ ]:
from mri_recon.utils import get_device, configure_backend
device = get_device("auto")          # cuda if present else cpu
configure_backend(device)            # tf32 + cudnn autotune on ampere+
print("using device:", device)

## 5. train on synthetic data (runs on any runtime)

no download needed trains the unet baseline and the unrolled centerpiece on synthetic phantoms in a few minutes proves the full pytorch path on the accelerator

In [ ]:
# unet baseline
!python -m mri_recon.cli train --config configs/synthetic.yaml --set train.epochs=8
# unrolled centerpiece (the differentiator, fp32 complex fft)
!python -m mri_recon.cli train --config configs/synthetic.yaml \
  --set model.name=unrolled train.loss=ssim train.batch_size=4 \
        output.dir=results/synthetic_unrolled train.epochs=8
# big ampere+ gpu? add  --set train.amp=true  to the unet line

## 6. evaluate + figures (synthetic)

In [ ]:
# all three methods, both checkpoints (list quoted so the shell doesnt glob it)
!python -m mri_recon.cli eval --config configs/eval.yaml \
  --set 'eval.methods=[zero_filled,unet,unrolled]' \
        eval.checkpoints.unet=results/synthetic/best.pt \
        eval.checkpoints.unrolled=results/synthetic_unrolled/best.pt
!python -m mri_recon.cli figures --config configs/eval.yaml \
  --set 'eval.methods=[zero_filled,unet,unrolled]' \
        eval.checkpoints.unet=results/synthetic/best.pt \
        eval.checkpoints.unrolled=results/synthetic_unrolled/best.pt

from IPython.display import Image, display
import glob
for p in sorted(glob.glob("results/eval/**/*.png", recursive=True)):
    print(p)
    display(Image(filename=p))

## 7. the real thing fastmri (knee single-coil val only)

download only knee_singlecoil_val (~15 gb) and split it by volume no need for the 72 gb train set save to drive so the download happens once see scripts/download_fastmri.md the signed nyu urls are time-limited tokens dont commit them anywhere

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/fastmri'   # persists across sessions
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
# paste the signed curl command from the nyu email downloading into drive
%cd $DRIVE
# !curl -C - "https://fastmri-dataset.s3.amazonaws.com/.../knee_singlecoil_val.tar.xz?..." --output knee_singlecoil_val.tar.xz
# !tar -xf knee_singlecoil_val.tar.xz       # -> singlecoil_val/
%cd $PROJECT_DIR

In [ ]:
# train unet @ 4x and unrolled @ 8x read data + write results to drive
DATA = f'{DRIVE}/singlecoil_val'
OUT = f'{DRIVE}/results'
!python -m mri_recon.cli train --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA --set output.dir=$OUT/knee_unet_4x
!python -m mri_recon.cli train --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA --set model.name=unrolled --set train.loss=ssim \
  --set train.batch_size=1 --set mask.accelerations='[8]' --set mask.center_fractions='[0.04]' \
  --set output.dir=$OUT/knee_unrolled_8x

In [ ]:
# evaluate all three methods at 4x and 8x render + show figures (from/to drive)
!python -m mri_recon.cli figures --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA --set output.dir=$OUT/eval \
  --set eval.methods='[zero_filled,unet,unrolled]' \
  --set eval.checkpoints.unet=$OUT/knee_unet_4x/best.pt \
  --set eval.checkpoints.unrolled=$OUT/knee_unrolled_8x/best.pt
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUT}/eval/**/*.png', recursive=True)):
    print(p); display(Image(filename=p))

## accelerator cheatsheet
- batch size scales with vram bump --set train.batch_size=... on a100/h100 keep 1 for the unrolled net on real (variable-size) fastmri k-space
- bf16 (a100/h100/l4) is the safe fast path fp16 for t4
- mixed precision is unet-only here the unrolled model complex fft stays fp32
- tpu use a gpu for this project (complex fft isnt xla-friendly)